In [1]:
!pip install scapy

In [2]:
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])


install("scapy")


from scapy.all import sniff, IP, TCP, UDP, ICMP, Raw, Ether, ARP
from datetime import datetime
import textwrap

In [3]:
PACKET_COUNT   = 30
INTERFACE      = None
SHOW_PAYLOAD   = True
LOG_TO_FILE    = True
LOG_FILE       = "sniffer_log.txt"

In [4]:
PROTOCOL_MAP = {1: "ICMP", 6: "TCP", 17: "UDP", 2: "IGMP", 89: "OSPF"}

def get_protocol(num):
    return PROTOCOL_MAP.get(num, f"OTHER({num})")

def get_tcp_flags(flags):
    flag_map = {"S": "SYN", "A": "ACK", "F": "FIN", "R": "RST",
                "P": "PSH", "U": "URG", "E": "ECE", "C": "CWR"}
    return " | ".join(v for k, v in flag_map.items() if k in str(flags)) or "NONE"

def safe_payload(raw_bytes, limit=80):
    try:
        text = raw_bytes[:limit].decode("utf-8", errors="replace")
        return repr(text)
    except Exception:
        return raw_bytes[:limit].hex()

In [5]:
packet_counter = [0]
log_lines      = []

SEPARATOR = "═" * 70

def analyse_packet(packet):
    packet_counter[0] += 1
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
    lines = []

    lines.append(f"\n{SEPARATOR}")
    lines.append(f"Packet #{packet_counter[0]:>4}    ⏱  {ts}")
    lines.append(SEPARATOR)

    if packet.haslayer(Ether):
        eth = packet[Ether]
        lines.append(f"  [ETHERNET]  Src MAC: {eth.src}  →  Dst MAC: {eth.dst}")

    if packet.haslayer(ARP):
        arp = packet[ARP]
        op  = "Request" if arp.op == 1 else "Reply"
        lines.append(f"  [ARP {op}]  {arp.psrc} ({arp.hwsrc})  →  {arp.pdst} ({arp.hwdst})")

    if packet.haslayer(IP):
        ip   = packet[IP]
        proto = get_protocol(ip.proto)
        lines.append(f"  [IP]  {ip.src}  →  {ip.dst}")
        lines.append(f"        Protocol : {proto}   TTL : {ip.ttl}   "
                             f"Length : {ip.len} bytes   ID : {ip.id}")

        if packet.haslayer(TCP):
            tcp = packet[TCP]
            lines.append(f"  [TCP]  Src Port: {tcp.sport}  →  Dst Port: {tcp.dport}")
            lines.append(f"         Flags : {get_tcp_flags(tcp.flags)}   "
                                   f"Seq : {tcp.seq}   Ack : {tcp.ack}   "
                                                           f"Win : {tcp.window}")

            service = {80: "HTTP", 443: "HTTPS", 21: "FTP", 22: "SSH",
                       23: "Telnet", 25: "SMTP", 53: "DNS", 3306: "MySQL",
                       3389: "RDP", 8080: "HTTP-ALT"}.get(tcp.dport) or \
                      {80: "HTTP", 443: "HTTPS", 21: "FTP", 22: "SSH",
                       23: "Telnet", 25: "SMTP", 53: "DNS", 3306: "MySQL",
                       3389: "RDP", 8080: "HTTP-ALT"}.get(tcp.sport)
            if service:
                lines.append(f"Likely Service : {service}")

        elif packet.haslayer(UDP):
            udp = packet[UDP]
            lines.append(f"  [UDP]  Src Port: {udp.sport}  →  Dst Port: {udp.dport}   "
                                   f"Length : {udp.len} bytes")
            if udp.dport == 53 or udp.sport == 53:
                lines.append("Likely Service : DNS")

        elif packet.haslayer(ICMP):
            icmp = packet[ICMP]
            icmp_types = {0: "Echo Reply", 3: "Destination Unreachable",
                          8: "Echo Request", 11: "Time Exceeded"}
            lines.append(f"  [ICMP]  Type: {icmp_types.get(icmp.type, icmp.type)}   "
                                   f"Code: {icmp.code}")

    if SHOW_PAYLOAD and packet.haslayer(Raw):
        raw_data = bytes(packet[Raw].load)
        preview  = safe_payload(raw_data)
        lines.append(f"  [PAYLOAD]  {len(raw_data)} bytes  →  {preview}")

    output = "\n".join(lines)
    print(output)
    if LOG_TO_FILE:
        log_lines.append(output)

In [6]:
import threading
import urllib.request

def generate_traffic():
    urls = [
        "http://example.com",
        "https://www.google.com",
        "https://www.python.org",
    ]
    for url in urls:
        try:
            urllib.request.urlopen(url, timeout=5)
        except Exception:
            pass

In [7]:
def main():
    print("╔══════════════════════════════════════════════════════════════════╗")
    print("║       CodeAlpha Cyber Security Internship — Task 1               ║")
    print("║                  Basic Network Sniffer                           ║")
    print("╚══════════════════════════════════════════════════════════════════╝")
    print(f"\n  ▶  Capturing {PACKET_COUNT} packets …")
    print("  ▶  Generating background HTTP/S traffic to populate capture …\n")

    t = threading.Thread(target=generate_traffic, daemon=True)
    t.start()

    sniff(
        prn       = analyse_packet,
        count     = PACKET_COUNT,
        iface     = INTERFACE,
        store     = False,
    )

    print(f"\n{SEPARATOR}")
    print(f"Capture complete — {packet_counter[0]} packets analysed.")
    print(SEPARATOR)

    if LOG_TO_FILE and log_lines:
        with open(LOG_FILE, "w") as f:
            f.write(f"CodeAlpha Network Sniffer Log — {datetime.now()}\n")
            f.write("\n".join(log_lines))
        print(f"\n Log saved to: {LOG_FILE}")
        print("      (Download from the Colab file browser on the left panel)")

    print("\nGitHub repo name : CodeAlpha_NetworkSniffer")
    print("LinkedIn post    : Share with #CodeAlpha #CyberSecurity\n")

In [8]:
if __name__ == "__main__":
    main()

╔══════════════════════════════════════════════════════════════════╗
║       CodeAlpha Cyber Security Internship — Task 1               ║
║                  Basic Network Sniffer                           ║
╚══════════════════════════════════════════════════════════════════╝

  ▶  Capturing 30 packets …
  ▶  Generating background HTTP/S traffic to populate capture …


══════════════════════════════════════════════════════════════════════
Packet #   1    ⏱  2026-05-23 17:38:54.614
══════════════════════════════════════════════════════════════════════
  [ETHERNET]  Src MAC: 02:42:18:49:5f:78  →  Dst MAC: 02:42:ac:1c:00:0c
  [IP]  169.254.169.254  →  172.28.0.12
        Protocol : UDP   TTL : 63   Length : 113 bytes   ID : 0
  [UDP]  Src Port: 53  →  Dst Port: 32976   Length : 93 bytes
Likely Service : DNS

══════════════════════════════════════════════════════════════════════
Packet #   2    ⏱  2026-05-23 17:38:54.619
══════════════════════════════════════════════════════════════════════